# M2 | CatBoost의 트리 수를 과거 시즌으로 선택하기

**한동희 · LG Aimers 9기 온라인 해커톤 · KBO 투구 제구 성공 확률 예측**

평가 직전 한 시즌의 결과로 트리 수를 고른 M1은 2024년 평가 모델에 트리 1개를 선택했습니다. M2에서는 평가 연도보다 앞선 여러 시즌의 평균 Brier Score를 기준으로 트리 수를 고른 뒤, 평가 직전까지의 데이터로 모델을 다시 학습했습니다. 이 노트북은 당시 M2의 CatBoost 학습 설정과 트리 수 선택 규칙을 확인할 수 있도록 정리한 공개본입니다.

## 실제 대회 데이터 내부 검증 결과

Brier Score는 실제 결과와 예측 확률 간 평균제곱오차이며, 낮을수록 좋습니다. 각 평가 투구는 이전 시즌들로 학습한 모델로 예측했습니다. 통합 Brier는 네 시즌의 투구를 합쳐 계산한 값입니다.

| 평가 시즌 | 트리 수 선택용 검증 시즌 | M2 트리 수 | 검증 투구 수 | M1 Brier | M2 Brier |
| --- | --- | ---: | ---: | ---: | ---: |
| 2021 | 2020 | 20 | 247,088 | 0.24705460 | 0.24682396 |
| 2022 | 2020–2021 | 40 | 247,472 | 0.24349204 | 0.24370823 |
| 2023 | 2020–2022 | 40 | 245,525 | 0.25323029 | 0.25288284 |
| 2024 | 2020–2023 | 40 | 253,507 | 0.24989241 | 0.24829435 |
| **통합** | | | **993,592** | **0.24841739** | **0.24792029** |

2022년에는 점수가 소폭 악화됐고 2024년에 가장 크게 개선됐습니다. 표의 수치는 당시 저장한 내부 검증 결과에서 옮겼습니다.

## 1. 실행 환경 선택

기본값은 **합성 데이터 CPU 데모**입니다. 그대로 실행하면 합성 투구 예제를 생성하고 같은 시간순 검증 절차로 CatBoost를 학습합니다. 데모는 축소된 입력과 후보 범위를 사용하며, 출력에는 데이터 종류가 표시됩니다.

대회 참여 권한에 맞게 확보한 원본 `train.csv`로 실행할 때는 GPU 런타임에서 본인 파일을 제공하고, 아래 `TRAIN_CSV`를 파일 경로로 지정하세요. 노트북에는 경기 데이터나 저장된 행별 예측이 포함되어 있지 않습니다.

[전체 실험 기록](https://github.com/dannyhan0609/lg-aimers-9-experiments) · [DACON 공식 대회 페이지](https://dacon.io/competitions/official/236743)

In [ ]:
from pathlib import Path
import importlib.metadata
import subprocess
import sys
from collections.abc import Iterable, Sequence

TRAIN_CSV = None  # 원본 데이터 실행 예시: Path("train.csv")

try:
    catboost_version = importlib.metadata.version("catboost")
except importlib.metadata.PackageNotFoundError:
    catboost_version = None

if catboost_version != "1.2.8":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost==1.2.8"])

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from IPython.display import display

## 2. 입력 변수와 학습 설정

원본 실험은 CatBoost 입력 46개와 시드 42를 사용했습니다. 입력에서 정답과 행 식별자를 제외하고, 투수 투구 수와 값이 같은 중복 변수를 제거합니다. 데모에서는 합성 시즌별 데이터를 만들어 원본 파일과 GPU가 없어도 학습 흐름을 점검할 수 있습니다.

In [ ]:
TARGET = 'control_success'
ID_COL = 'row_id'
DUPLICATE_COL = 'asof_pitcher_pitchmix_n'
FEATURES = [
    'season', 'game_month', 'game_dayofweek', 'inning', 'top_bottom', 'game_type',
    'balls_before', 'strikes_before', 'outs_before', 'run_top_before', 'run_bot_before',
    'run_total_before', 'score_diff_home', 'score_diff_pitcher_team', 'runner_on_1b',
    'runner_on_2b', 'runner_on_3b', 'num_runners_on', 'base_state',
    'home_win_expectancy', 'away_win_expectancy', 'li', 'pitcher_id', 'batter_id',
    'pitcher_hand', 'batter_hand', 'pitcher_team_id', 'batter_team_id', 'asof_pitcher_n',
    'asof_pitcher_success_rate', 'asof_pitcher_reverse_rate', 'asof_pitcher_middle_rate',
    'asof_pitcher_ball_rate', 'asof_pitcher_strike_rate',
    'asof_pitcher_prev1_game_success_rate', 'asof_pitcher_prev3_game_success_rate',
    'asof_pitcher_prev5_game_success_rate', 'asof_pitcher_prev1_game_middle_rate',
    'asof_pitcher_prev3_game_middle_rate', 'asof_pitcher_prev5_game_middle_rate',
    'asof_batter_n', 'asof_batter_success_rate', 'asof_batter_middle_rate',
    'asof_pitcher_fastball_rate', 'asof_pitcher_breaking_rate', 'asof_pitcher_offspeed_rate',
]
CAT_FEATURES = [
    'game_month', 'game_dayofweek', 'top_bottom', 'game_type', 'base_state',
    'pitcher_id', 'batter_id', 'pitcher_hand', 'batter_hand',
    'pitcher_team_id', 'batter_team_id',
]
assert len(FEATURES) == 46 and len(set(FEATURES)) == 46

TREE_CANDIDATES = [1, 5, 10, 20, 40, 80, 120, 160, 240, 320]
HISTORY_VALIDATION_YEARS = (2020, 2021, 2022, 2023)
EVALUATION_YEARS = (2021, 2022, 2023, 2024)
SEED = 42

BASE_PARAMS = dict(
    loss_function='Logloss',
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=10.0,
    random_seed=SEED,
    task_type='GPU',
    devices='0',
    bootstrap_type='Bayesian',
    random_strength=1.0,
    allow_writing_files=False,
    verbose=False,
)

In [ ]:
if TRAIN_CSV is None:
    rng = np.random.default_rng(SEED)
    rows_per_season = 160
    seasons = np.repeat(np.arange(2019, 2025), rows_per_season)
    row_count = len(seasons)
    balls = rng.integers(0, 4, row_count)
    strikes = rng.integers(0, 3, row_count)
    hands = rng.choice(["L", "R"], row_count)
    logits = 0.35 - 0.45 * balls + 0.30 * strikes + 0.15 * (hands == "L")
    logits += 0.07 * (seasons - 2019)
    probabilities = 1 / (1 + np.exp(-logits))
    train = pd.DataFrame({
        "season": seasons,
        "balls_before": balls,
        "strikes_before": strikes,
        "pitcher_hand": hands,
        TARGET: rng.binomial(1, probabilities),
    })
    features = ["season", "balls_before", "strikes_before", "pitcher_hand"]
    categorical_features = ["pitcher_hand"]
    tree_candidates = [1, 5, 10, 20, 40]
    params = {**BASE_PARAMS, "task_type": "CPU", "depth": 4, "thread_count": 2}
    params.pop("devices")
    data_label = "합성 데이터 CPU 데모"
else:
    from catboost.utils import get_gpu_device_count

    if get_gpu_device_count() < 1:
        raise RuntimeError("원본 실험 설정을 사용하려면 Colab GPU 런타임이 필요합니다.")
    train = pd.read_csv(TRAIN_CSV, usecols=[ID_COL, *FEATURES, DUPLICATE_COL, TARGET],
                        low_memory=False)
    if train.shape != (1_475_092, 49):
        raise ValueError(f"원본 M2와 입력 데이터 크기가 다릅니다: {train.shape}")
    if not train["asof_pitcher_n"].equals(train[DUPLICATE_COL]):
        raise ValueError("원본 M2에서 확인한 중복 변수 관계가 성립하지 않습니다.")
    train.drop(columns=[DUPLICATE_COL], inplace=True)
    features = FEATURES
    categorical_features = CAT_FEATURES
    tree_candidates = TREE_CANDIDATES
    params = BASE_PARAMS
    data_label = "대회 데이터 내부 검증"

for column in categorical_features:
    train[column] = train[column].fillna("__MISSING__").astype(str).astype("category")
for column in [name for name in features if name not in categorical_features]:
    train[column] = pd.to_numeric(train[column], errors="coerce", downcast="float")
train[TARGET] = train[TARGET].astype("int8")

print(f"{data_label} | 입력 {len(train):,}건 | 변수 {len(features)}개")

## 3. Brier 계산과 트리 수 선택

후보별 평균에는 평가 연도보다 앞선 검증 결과만 넣습니다. 예를 들어 2024년 모델의 트리 수를 고를 때는 2020–2023년 검증 결과를 사용합니다. 후보들의 평균 점수가 같으면 트리 수가 적은 쪽을 고릅니다.

In [ ]:
def score_metrics(y_true: Iterable[float], probability: Iterable[float]) -> dict[str, float]:
    """Calculate Brier score and the competition-style score."""

    y = np.asarray(list(y_true), dtype=np.float64)
    p = np.asarray(list(probability), dtype=np.float64)
    if y.size == 0 or y.shape != p.shape:
        raise ValueError("정답과 예측은 길이가 같은 비어 있지 않은 1차원 배열이어야 합니다.")
    if not np.isfinite(y).all() or not np.isfinite(p).all():
        raise ValueError("정답과 예측에는 유한값만 사용할 수 있습니다.")

    brier = float(np.mean((p - y) ** 2))
    rate = float(np.mean(y))
    baseline = float(rate * (1.0 - rate))
    raw_score = float(100_000 * (1.0 - brier / baseline)) if baseline > 0 else 0.0
    return {
        "actual_success_rate": rate,
        "prediction_mean": float(np.mean(p)),
        "brier_score": brier,
        "baseline_brier": baseline,
        "official_score_raw": raw_score,
        "official_score": max(0.0, raw_score),
    }

def build_candidate_summary(
    history: pd.DataFrame,
    evaluation_year: int,
    candidates: Sequence[int],
) -> pd.DataFrame:
    """Summarize candidate Brier scores using only years before evaluation_year."""

    required = {"validation_year", "tree_count", "brier_score"}
    missing_columns = required.difference(history.columns)
    if missing_columns:
        raise ValueError(f"필수 열이 없습니다: {sorted(missing_columns)}")
    if not candidates or len(set(candidates)) != len(candidates):
        raise ValueError("후보 트리 목록은 중복이 없는 비어 있지 않은 목록이어야 합니다.")

    past = history.loc[
        history["validation_year"].lt(evaluation_year)
        & history["tree_count"].isin(candidates),
        ["validation_year", "tree_count", "brier_score"],
    ].copy()
    years = sorted(int(year) for year in past["validation_year"].unique())
    if not years:
        raise ValueError("평가 연도 이전의 후보 검증 결과가 없습니다.")
    if past.duplicated(["validation_year", "tree_count"]).any():
        raise ValueError("연도와 후보 트리 조합은 하나의 Brier만 가져야 합니다.")

    expected = pd.MultiIndex.from_product(
        [years, list(candidates)], names=["validation_year", "tree_count"]
    )
    actual = pd.MultiIndex.from_frame(past[["validation_year", "tree_count"]])
    if not expected.isin(actual).all():
        raise ValueError("후보별 과거 연도 검증 결과가 완전하지 않습니다.")
    if not np.isfinite(past["brier_score"].to_numpy(dtype=np.float64)).all():
        raise ValueError("Brier에는 유한값만 사용할 수 있습니다.")

    grouped = past.groupby("tree_count", sort=False)["brier_score"]
    summary = grouped.agg(mean_brier="mean", std_brier="std").reindex(candidates).reset_index()
    summary["history_years"] = ",".join(map(str, years))
    summary["history_year_count"] = len(years)
    return summary[
        ["tree_count", "history_years", "history_year_count", "mean_brier", "std_brier"]
    ]

def select_tree_count(summary: pd.DataFrame) -> int:
    """Select the lowest mean Brier, breaking exact ties toward fewer trees."""

    required = {"tree_count", "mean_brier"}
    if summary.empty or not required.issubset(summary.columns):
        raise ValueError("후보 요약에는 tree_count와 mean_brier가 필요합니다.")
    if not np.isfinite(summary["mean_brier"].to_numpy(dtype=np.float64)).all():
        raise ValueError("평균 Brier에는 유한값만 사용할 수 있습니다.")

    best = summary.sort_values(["mean_brier", "tree_count"], kind="stable").iloc[0]
    return int(best["tree_count"])

## 4. 과거 시즌에서 후보별 Brier 계산

각 과거 검증 시즌보다 이전에 나온 투구로 CatBoost를 학습합니다. 후보 중 최대 트리 수까지 학습한 뒤 `ntree_end`를 바꿔, 같은 모델의 앞쪽 트리만 사용한 후보별 예측을 구합니다.

In [ ]:
def make_pool(mask):
    return Pool(train.loc[mask, features], train.loc[mask, TARGET],
                cat_features=categorical_features)

candidate_history_rows = []

for validation_year in HISTORY_VALIDATION_YEARS:
    train_mask = train["season"].lt(validation_year)
    validation_mask = train["season"].eq(validation_year)
    history_train_pool = make_pool(train_mask)
    history_validation_pool = make_pool(validation_mask)
    history_model = CatBoostClassifier(**params, iterations=max(tree_candidates))
    history_model.fit(history_train_pool)

    for tree_count in tree_candidates:
        probability = history_model.predict_proba(
            history_validation_pool, ntree_end=tree_count
        )[:, 1]
        metrics = score_metrics(train.loc[validation_mask, TARGET], probability)
        candidate_history_rows.append({
            "validation_year": validation_year,
            "tree_count": tree_count,
            "brier_score": metrics["brier_score"],
        })

candidate_history = pd.DataFrame(candidate_history_rows)
assert len(candidate_history) == len(HISTORY_VALIDATION_YEARS) * len(tree_candidates)
display(candidate_history.pivot(index="tree_count", columns="validation_year",
                                values="brier_score"))

## 5. 트리 수를 고른 뒤 해당 평가 시즌 검증

평가 시즌마다 트리 수를 결정한 뒤 이전 시즌 전체로 최종 모델을 재학습합니다. 이후 해당 시즌의 예측 확률과 정답을 비교하고, 네 시즌의 예측을 합쳐 통합 Brier를 계산합니다. 출력은 현재 실행의 시즌별 집계와 통합 점수입니다.

In [ ]:
fold_results = []
observed_parts = []
predicted_parts = []

for evaluation_year in EVALUATION_YEARS:
    summary = build_candidate_summary(candidate_history, evaluation_year, tree_candidates)
    selected_trees = select_tree_count(summary)
    history_years = summary["history_years"].iloc[0]

    final_train_mask = train["season"].lt(evaluation_year)
    outer_validation_mask = train["season"].eq(evaluation_year)
    final_train_pool = make_pool(final_train_mask)
    outer_validation_pool = make_pool(outer_validation_mask)
    final_model = CatBoostClassifier(**params, iterations=selected_trees)
    final_model.fit(final_train_pool)
    probability = final_model.predict_proba(outer_validation_pool)[:, 1]
    observed = train.loc[outer_validation_mask, TARGET].to_numpy()
    metrics = score_metrics(observed, probability)

    fold_results.append({
        "평가 시즌": evaluation_year,
        "트리 수 선택용 검증 시즌": history_years,
        "선택한 트리 수": selected_trees,
        "검증 투구 수": len(observed),
        "Brier Score": metrics["brier_score"],
    })
    observed_parts.append(observed)
    predicted_parts.append(probability)

observed_oof = np.concatenate(observed_parts)
predicted_oof = np.concatenate(predicted_parts)
if TRAIN_CSV is not None and len(observed_oof) != 993_592:
    raise ValueError(f"원본 M2와 검증 투구 수가 다릅니다: {len(observed_oof):,}")

results = pd.DataFrame(fold_results)
oof_metrics = score_metrics(observed_oof, predicted_oof)
print(data_label)
display(results)
print(f"통합 검증 {len(observed_oof):,}건 | Brier Score {oof_metrics['brier_score']:.8f}")